In [1]:
import os
import pandas as pd                                         # for data manipulation
import torch                                                # for tensor computations
from transformers import GPT2LMHeadModel, GPT2Tokenizer     # for GPT-2 model and tokenizer
from tqdm import tqdm                                       # for progress bar
import numpy as np                                          # for numerical operations
import nltk                                                 # Natural Language Toolkit
from nltk.tokenize import sent_tokenize                     # for sentence tokenization
import re                                                   # for regex operations
import spacy                                                # for NLP processing
from empath import Empath                                   # for LIWC analysis
from sentence_transformers import SentenceTransformer, util  # for semantic analysis
import collections                                          # for counting duplicates
import random                                               # for random seed setting

import gc
from transformers import AutoModel, AutoTokenizer

# Download necessary NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ==================== REPRODUCIBILITY SETUP ====================
SEED = 999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"[REPRODUCIBILITY] Fixed seed set to: {SEED}")
print(f"[REPRODUCIBILITY] All random operations will use SEED={SEED}")

[REPRODUCIBILITY] Fixed seed set to: 999
[REPRODUCIBILITY] All random operations will use SEED=999


# Feature Engineering Pipeline

In [2]:

# ==================== FEATURE EXTRACTION HELPERS ====================

def is_valid_text(text):
    """Check if text is a non-empty string."""
    return isinstance(text, str) and len(text.strip()) > 0


def calculate_perplexity(text, model, tokenizer, device, stride=512):
    """Calculate perplexity using GPT-2 with a sliding window."""
    if not is_valid_text(text):
        return None

    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = int(1e30)
    
    encodings = tokenizer(text, return_tensors="pt", truncation=False)
    tokenizer.model_max_length = original_max_length
    
    seq_len = encodings.input_ids.size(1)
    max_length = model.config.n_positions

    nlls = []
    prev_end_loc = 0

    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            with torch.amp.autocast(device_type=device):
                outputs = model(input_ids, labels=target_ids)
                neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    return torch.exp(torch.stack(nlls).sum() / seq_len).item()


def process_perplexity(texts_list, model, tokenizer, device):
    """Batch perplexity computation."""
    return [calculate_perplexity(text, model, tokenizer, device) for text in tqdm(texts_list, desc="Calculating Perplexity")]


def calculate_burstiness(text):
    """Sentence-length variance."""
    if not is_valid_text(text):
        return 0.0
    sentences = sent_tokenize(text)
    lengths = np.array([len(s.split()) for s in sentences if len(s.split()) > 0])
    if len(lengths) < 2:
        return 0.0
    return float(np.std(lengths))


def process_burstiness_list(texts_list):
    """Vectorized burstiness calculation using NumPy operations."""
    results = []
    for text in tqdm(texts_list, desc="Calculating Burstiness"):
        if is_valid_text(text):
            sentences = sent_tokenize(text)
            lengths = np.array([len(s.split()) for s in sentences if len(s.split()) > 0])
            if len(lengths) >= 2:
                results.append(float(np.std(lengths)))
            else:
                results.append(0.0)
        else:
            results.append(0.0)
    return results


def calculate_ttr(text):
    """Type-Token Ratio (lexical diversity)."""
    if not is_valid_text(text):
        return 0.0
    words = re.findall(r"\w+", text.lower())
    if not words:
        return 0.0
    return float(len(set(words)) / len(words))


def process_ttr_list(texts_list):
    """Vectorized TTR calculation using NumPy operations."""
    results = np.zeros(len(texts_list), dtype=np.float32)
    for idx, text in enumerate(tqdm(texts_list, desc="Calculating Lexical Diversity (TTR)")):
        if is_valid_text(text):
            words = re.findall(r"\w+", text.lower())
            if words:
                results[idx] = float(len(set(words)) / len(words))
    return results


def extract_stylometric_features(texts_series):
    """Vectorized extraction of stylometric features for a pandas Series."""
    complex_ratios = np.zeros(len(texts_series), dtype=np.float32)
    avg_word_lens = np.zeros(len(texts_series), dtype=np.float32)
    avg_sent_lens = np.zeros(len(texts_series), dtype=np.float32)
    
    for idx, text in enumerate(tqdm(texts_series, desc="Extracting Stylometric Features")):
        if is_valid_text(text):
            words = re.findall(r"\w+", text.lower())
            sentences = sent_tokenize(text)
            if words and sentences:
                avg_word_lens[idx] = float(np.mean([len(w) for w in words]))
                avg_sent_lens[idx] = float(len(words) / len(sentences))
                complex_ratios[idx] = float(len([w for w in words if len(w) > 6]) / len(words))
    
    return complex_ratios, avg_word_lens, avg_sent_lens


def calculate_uid(text, model, tokenizer, device, max_length=1024, stride=512):
    """Uniform Information Density (std of surprisal) with sliding window for long texts."""
    if not is_valid_text(text):
        return 0.0
    
    # Temporarily increase model_max_length to suppress warnings
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = int(1e30)
    
    inputs = tokenizer(text, return_tensors="pt", truncation=False)
    tokenizer.model_max_length = original_max_length
    
    input_ids = inputs["input_ids"]
    seq_len = input_ids.size(1)
    
    if seq_len < 2:
        return 0.0
    
    all_surprisals = []
    
    # Sliding window for long texts
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        
        chunk_ids = input_ids[:, begin_loc:end_loc].to(device)
        
        if chunk_ids.shape[1] < 2:
            continue
            
        with torch.no_grad():
            with torch.amp.autocast(device_type=device, enabled=(device == "cuda")):
                outputs = model(chunk_ids, labels=chunk_ids)
                logits = outputs.logits
        
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = chunk_ids[:, 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        surprisal = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        all_surprisals.append(surprisal)
        
        if end_loc >= seq_len:
            break
    
    if not all_surprisals:
        return 0.0
    
    # Concatenate all surprisals and compute std - VECTORIZED
    combined_surprisals = torch.cat(all_surprisals)
    return float(torch.std(combined_surprisals).item())


def extract_empath(df, text_column, lexicon, categories=None):
    """Vectorized Empath LIWC features using batch processing."""
    print(f"Processing Empath features for: {text_column} (categories: {categories})...")
    feats = []
    
    # Batch process texts for efficiency
    batch_size = 32
    texts = df[text_column].tolist()
    
    for i in tqdm(range(0, len(texts), batch_size), desc=f"{text_column} - Empath"):
        batch_texts = texts[i:i+batch_size]
        for text in batch_texts:
            if is_valid_text(text):
                res = lexicon.analyze(text, categories=categories, normalize=True) or {}
            else:
                res = {}
            res_filled = {cat: res.get(cat, 0.0) for cat in categories}
            feats.append(res_filled)
    
    temp_df = pd.DataFrame(feats).add_prefix(f"{text_column}_")
    return temp_df


def calculate_avg_syntax_depth(text, nlp):
    """Average parse-tree depth per sentence."""
    if not is_valid_text(text):
        return 0.0
    doc = nlp(text)
    def walk(node, depth):
        if node.n_lefts + node.n_rights == 0:
            return depth
        return max(walk(child, depth + 1) for child in node.children)
    depths = [walk(sent.root, 1) for sent in doc.sents]
    return float(np.mean(depths)) if depths else 0.0


def get_semantic_features_batch(texts_list, semantic_model, device):
    """Vectorized semantic feature extraction using batch encoding."""
    results = []
    
    # Filter valid texts and keep track of indices
    valid_texts = [(i, text) for i, text in enumerate(texts_list) if is_valid_text(text)]
    
    if not valid_texts:
        return np.zeros((len(texts_list), 2), dtype=np.float32)
    
    # Use batch encoding from semantic_model (more efficient than single texts)
    all_means = np.zeros(len(texts_list), dtype=np.float32)
    all_stds = np.zeros(len(texts_list), dtype=np.float32)
    
    for idx, text in tqdm(valid_texts, desc="Semantic Features", total=len(valid_texts)):
        sentences = [s for s in sent_tokenize(text) if len(s.split()) > 1]
        if len(sentences) >= 2:
            with torch.no_grad():
                # Batch encode all sentences at once
                embeddings = semantic_model.encode(sentences, convert_to_tensor=True, device=device)
            # Vectorized similarity computation
            sims = torch.tensor([util.cos_sim(embeddings[i], embeddings[i+1]).item() 
                                 for i in range(len(embeddings) - 1)])
            all_means[idx] = float(torch.mean(sims).item())
            all_stds[idx] = float(torch.std(sims).item())
    
    return all_means, all_stds


# BERT

In [3]:
# ==================== BERT CONFIGURATION ====================

MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 8
MAX_LENGTH = 512
CHUNK_OVERLAP = 50

# Global model and tokenizer for BERT (loaded once)
bert_tokenizer = None
bert_model = None
bert_device = None

print(f"[CONFIG] BERT Model: {MODEL_NAME} (768-dim embeddings)")
print(f"[CONFIG] Max Length: {MAX_LENGTH} tokens, Overlap: {CHUNK_OVERLAP} tokens, Batch Size: {BATCH_SIZE}")

def initialize_bert_model(model_name=MODEL_NAME):
    
    """Initialize BERT model and tokenizer once."""
    global bert_tokenizer, bert_model, bert_device
    
    if bert_model is None:
        print(f"\nLoading BERT model: {model_name}")
        bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
        bert_model = AutoModel.from_pretrained(model_name)
        
        # Freeze model weights
        for param in bert_model.parameters():
            param.requires_grad = False
        
        bert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        bert_model.to(bert_device)
        bert_model.eval()
        print(f"Model loaded on device: {bert_device}")


def chunk_text_by_tokens(text, max_length=MAX_LENGTH, overlap=CHUNK_OVERLAP):
    """
    Split long text into overlapping chunks based on token count.
    Ensures chunks never exceed max_length tokens.
    Returns list of token tensors.
    """
    encoded = bert_tokenizer.encode_plus(
        text,
        add_special_tokens=False,
        truncation=False,
        return_tensors=None
    )
    token_ids = encoded['input_ids']
    
    # If text fits in one chunk, return it
    if len(token_ids) <= max_length - 2:  # -2 because of special tokens
        inputs = bert_tokenizer.encode_plus(
            text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return [inputs]
    
    # Split tokens into overlapping chunks
    chunks = []
    chunk_size = max_length - 2
    stride = chunk_size - overlap
    
    for i in range(0, len(token_ids), stride):
        chunk_ids = token_ids[i:i + chunk_size]
        
        # Convert back to text
        chunk_text = bert_tokenizer.decode(chunk_ids, skip_special_tokens=True)
        
        chunk_input = bert_tokenizer.encode_plus(
            chunk_text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        chunks.append(chunk_input)
        
        if i + chunk_size >= len(token_ids):
            break
    
    return chunks if chunks else [bert_tokenizer.encode_plus(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )]


def get_bert_embeddings(text_list, batch_size=BATCH_SIZE):
    """
    Extract BERT embeddings for long texts using token-based chunking.
    
    Args:
        text_list: List of text strings (can be very long)
        batch_size: Number of chunks to process at once    
    Returns:
        numpy array of shape (n_texts, 768) containing embeddings
    """
    all_embeddings = []
    
    for text in tqdm(text_list, desc="Extracting BERT Embeddings"):
        try:
            chunks = chunk_text_by_tokens(str(text))
            chunk_embeddings = []
            
            for i in range(0, len(chunks), batch_size):
                batch_chunks = chunks[i:i+batch_size]
                
                # Stack inputs safely
                input_ids = torch.cat([c['input_ids'] for c in batch_chunks]).to(bert_device)
                attention_mask = torch.cat([c['attention_mask'] for c in batch_chunks]).to(bert_device)
                
                with torch.no_grad():
                    outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
                    cls_embs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                    chunk_embeddings.append(cls_embs)
            
            all_chunk_embs = np.vstack(chunk_embeddings)
            text_embedding = np.mean(all_chunk_embs, axis=0)
            all_embeddings.append(text_embedding)
            
        except Exception as e:
            print(f"Error processing text: {str(e)[:100]}") 
            all_embeddings.append(np.zeros(768))  # 768 is BERT hidden size
    
    return np.array(all_embeddings)


def add_bert_features(df, output_dir,text_column='Text'):
    """
    Add BERT embeddings to a DataFrame.
    
    Args:
        df: Input DataFrame
        text_column: Name of the column containing text (default: 'Text')
    
    Returns:
        DataFrame with BERT features added
    """
    # Initialize model on first call
    initialize_bert_model()
    
    texts = df[text_column].tolist()
    
    print(f"\n{'='*60}")
    print("Adding BERT embeddings...")
    print(f"{'='*60}")
    
    bert_embeddings = get_bert_embeddings(texts)
    
    bert_columns = [f'bert_{i}' for i in range(bert_embeddings.shape[1])]
    df_bert = pd.DataFrame(bert_embeddings, columns=bert_columns)
    df_combined = pd.concat([df.reset_index(drop=True), df_bert], axis=1)
    
    print(f"\nBERT embeddings added - shape: {df_combined.shape}")
    
    # Cleanup
    del bert_embeddings, df_bert
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("BERT features added!")
    
    output_path = os.path.join(output_dir, "DB_final_with_BERT.csv")
    df_combined.to_csv(output_path, index=False)

[CONFIG] BERT Model: bert-base-uncased (768-dim embeddings)
[CONFIG] Max Length: 512 tokens, Overlap: 50 tokens, Batch Size: 8


# Main function to process the dataset

In [4]:
# ==================== MAIN PIPELINE ====================

def process_dataset(input_dataset: str, output_file: str, human_col_name: str = 'Human_story') -> pd.DataFrame:
    """Run full feature pipeline for the provided writers and save everything (including BERT) to a single CSV.
    
    Supports resumable checkpoints - if a checkpoint exists, resumes from the last completed step.

    input_dataset: CSV path containing at least 'Writers' and 'Article' columns
    output_file: full path to the final CSV file (e.g., 'results/processed_data_with_BERT.csv')
    human_col_name: name of the human-written column
    """
    output_dir = os.path.dirname(output_file)

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    checkpoint_file = output_file.replace('.csv', '_checkpoint.csv') if output_file.endswith('.csv') else output_file + '_checkpoint.csv'

    np.random.seed(999)
    torch.manual_seed(999)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(999)

    # ==================== CHECKPOINT RESUMPTION ====================
    if os.path.exists(checkpoint_file):
        print(f"[RESUME] Found checkpoint file. Loading and resuming from last step...")
        df = pd.read_csv(checkpoint_file)
        
        # Determine which step to resume from based on columns present
        if 'bert_0' in df.columns:
            print("[RESUME] All steps completed! Final file exists, proceeding to save...")
            df.to_csv(output_file, index=False)
            print(f"Final dataset saved to: {output_file}")
            return df
        elif 'Text' in df.columns and 'is_AI' in df.columns:
            print("[RESUME] Resuming from step [11/11] - BERT Embeddings...")
            train_df = df.copy()
            goto_bert = True
        elif 'semantic_std' in df.columns:
            print("[RESUME] Resuming from step [10/11] - Formatting dataset...")
            goto_formatting = True
        elif 'semantic_mean' in df.columns:
            print("[RESUME] Resuming from step [9/11] - Semantic Consistency (already done)...")
            goto_semantic = False
        elif 'syntax_depth' in df.columns:
            print("[RESUME] Resuming from step [9/11] - Semantic Consistency...")
            goto_semantic = True
        elif 'Art' in [c for c in df.columns if 'Art' in c] or 'beauty' in df.columns:
            print("[RESUME] Resuming from step [8/11] - Syntax Tree Depth...")
            goto_syntax = True
        else:
            print("[RESUME] Resuming from an earlier step...")
            goto_syntax = True
        
        print(f"Checkpoint loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
    else:
        print("[1/11] Loading dataset...")
        df = pd.read_csv(input_dataset)
        
        if 'Writers' not in df.columns or 'Article' not in df.columns:
            raise ValueError(f"Dataset must contain 'Writers' and 'Article' columns. Found: {df.columns.tolist()}")

        writers = df["Writers"].unique().tolist()
            
        print(f"Dataset shape after filtering: {df.shape}")
        print(f"Writers: {writers}")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")

        # Load GPT-2 once for both perplexity and UID to save time/memory
        model_id = 'gpt2'
        gpt_tokenizer = GPT2Tokenizer.from_pretrained(model_id)
        gpt_model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
        gpt_model = gpt_model.half() if device == "cuda" else gpt_model
        gpt_model.eval()

        # ==================== PERPLEXITY ====================
        print("[2/11] Calculating Perplexity...")
        df['ppl'] = process_perplexity(df['Article'].tolist(), gpt_model, gpt_tokenizer, device)
        df.to_csv(checkpoint_file, index=False)

        # ==================== BURSTINESS ====================
        print("[3/11] Calculating Burstiness...")
        df['burstiness'] = process_burstiness_list(df['Article'].tolist())
        df.to_csv(checkpoint_file, index=False)

        # ==================== TYPE-TOKEN RATIO (TTR) ====================
        print("[4/11] Calculating Type-Token Ratio (TTR)...")
        df['ttr'] = process_ttr_list(df['Article'].tolist())
        df.to_csv(checkpoint_file, index=False)

        # ==================== STYLOMETRY - VECTORIZED ====================
        print("[5/11] Calculating Stylometric Features...")
        complex_ratios, avg_word_lens, avg_sent_lens = extract_stylometric_features(df['Article'])
        df['complex'] = complex_ratios
        df['avg_word_len'] = avg_word_lens
        df['avg_sent_len'] = avg_sent_lens
        df.to_csv(checkpoint_file, index=False)

        # ==================== UNIFORM INFORMATION DENSITY (UID) ====================
        print("[6/11] Calculating Uniform Information Density (UID)...")
        df['uid'] = [calculate_uid(x, gpt_model, gpt_tokenizer, device) for x in tqdm(df['Article'], desc="Calculating UID")]
        df.to_csv(checkpoint_file, index=False)
        
        del gpt_model, gpt_tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ==================== LIWC (EMPATH) ====================
        print("[7/11] Calculating LIWC (Empath) Features...")
        USED_EMPATH_CATEGORIES = [
            'gain', 'beauty', 'government', 'urban', 'art',
            'help', 'optimism', 'strength', 'love', 'traveling',
        ]
        lexicon = Empath()
        empath_frames = extract_empath(df, 'Article', lexicon, categories=USED_EMPATH_CATEGORIES)
        # Rename empath columns to drop 'Article_' prefix to match previous format if desired
        empath_frames.columns = [col.replace('Article_', '') for col in empath_frames.columns]
        df = pd.concat([df, empath_frames], axis=1)
        df.to_csv(checkpoint_file, index=False)

        goto_syntax = True

    # ==================== SYNTAX TREE DEPTH ====================
    if 'goto_syntax' in locals() and goto_syntax or ('syntax_depth' not in df.columns):
        print("[8/11] Calculating Syntax Tree Depth...")
        spacy.prefer_gpu()
        nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
        
        # Syntax depth calculation with proper handling of invalid texts
        syntax_depths = []
        texts_list = df['Article'].tolist()
        
        # Helper function for syntax depth calculation
        def get_syntax_depth(doc):
            def walk(node, depth):
                if node.n_lefts + node.n_rights == 0:
                    return depth
                return max(walk(child, depth + 1) for child in node.children)
            depths = [walk(sent.root, 1) for sent in doc.sents]
            return float(np.mean(depths)) if depths else 0.0
        
        # Process texts with validation (avoid multiprocessing issues with NaN values)
        for text in tqdm(texts_list, desc="Syntax Depth"):
            if is_valid_text(text):
                doc = nlp(text)
                syntax_depths.append(get_syntax_depth(doc))
            else:
                syntax_depths.append(0.0)
        
        df['syntax_depth'] = syntax_depths
        df.to_csv(checkpoint_file, index=False)

    # ==================== SEMANTIC CONSISTENCY - VECTORIZED ====================
    if 'semantic_mean' not in df.columns:
        print("[9/11] Calculating Semantic Consistency...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        semantic_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)
        means, stds = get_semantic_features_batch(df['Article'].tolist(), semantic_model, device)
        df['semantic_mean'] = means
        df['semantic_std'] = stds
        df.to_csv(checkpoint_file, index=False)

        del semantic_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("[9/11] Semantic Consistency (already completed, skipping)...")

    # ==================== FORMATTING ====================
    if 'is_AI' not in df.columns:
        print("[10/11] Formatting dataset...")
        
        # Rename columns to match expected output
        df = df.rename(columns={'Article': 'Text', 'Writers': 'Writer'})
        # Vectorized is_AI assignment
        df['is_AI'] = (df['Writer'].str.lower() != human_col_name.lower()).astype(int)

        train_df = df.copy()
        print(f"Rows before dropna: {len(train_df)}")
        train_df.dropna(inplace=True)
        train_df.reset_index(drop=True, inplace=True)
        print(f"Rows after dropna: {len(train_df)}")
        train_df.to_csv(checkpoint_file, index=False)
    else:
        print("[10/11] Formatting (already completed, skipping)...")
        train_df = df.copy()

    # ==================== BERT EMBEDDINGS ====================
    if 'bert_0' not in df.columns:
        print("[11/11] Extracting BERT Embeddings...")
        initialize_bert_model()
        bert_embeddings = get_bert_embeddings(train_df['Text'].tolist())
        
        bert_columns = [f'bert_{i}' for i in range(bert_embeddings.shape[1])]
        df_bert = pd.DataFrame(bert_embeddings, columns=bert_columns, index=train_df.index)
        train_df = pd.concat([train_df, df_bert], axis=1)
        
        del bert_embeddings, df_bert
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("[11/11] BERT Embeddings (already completed, skipping)...")

    print("[DONE] Saving final unified dataset...")
    train_df.to_csv(output_file, index=False)

    # Clean up checkpoint
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
        print(f"Removed temporary checkpoint file.")

    print("" + "=" * 80)
    print("PIPELINE COMPLETE!")
    print("=" * 80)
    print(f"Final DataFrame Shape: {train_df.shape}")
    print(f"Total Samples: {len(train_df)}")
    print(f"Dataset successfully saved to: {output_file}")
    print("=" * 80)
    return train_df

In [5]:
input_dir=os.path.join(os.getcwd(), "data/need_processing")
output_dir = os.path.join(os.getcwd(), "data/processed")
for csv in os.listdir(input_dir)[::-1]:
    if csv.endswith(".csv"):
        input = os.path.join(input_dir, csv)
        output = os.path.join(output_dir, f"processed_{csv}")
        human_col_name = "Human_story"
        process_dataset(input_dataset=input, output_file=output, human_col_name=human_col_name)

[RESUME] Found checkpoint file. Loading and resuming from last step...
[RESUME] Resuming from step [8/11] - Syntax Tree Depth...
Checkpoint loaded: 51247 rows, 19 columns
Using device: cuda
[8/11] Calculating Syntax Tree Depth...


/home/basel/Work/python/ai_text_detection/.venv/lib/python3.11/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
Syntax Depth: 100%|██████████| 51247/51247 [37:33<00:00, 22.74it/s] 


[9/11] Calculating Semantic Consistency...


Semantic Features:   0%|          | 1/51181 [00:00<9:03:42,  1.57it/s]/tmp/ipykernel_5760/888588773.py:228: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  all_stds[idx] = float(torch.std(sims).item())
Semantic Features: 100%|██████████| 51181/51181 [11:30<00:00, 74.17it/s] 


[10/11] Formatting dataset...
Rows before dropna: 51247
Rows after dropna: 50601
[11/11] Extracting BERT Embeddings...

Loading BERT model: bert-base-uncased
Model loaded on device: cuda


...

[9/11] Calculating Semantic Consistency...


Semantic Features:   0%|          | 44/10242 [00:00<02:59, 56.78it/s]/tmp/ipykernel_5760/888588773.py:228: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  all_stds[idx] = float(torch.std(sims).item())
Semantic Features: 100%|██████████| 10242/10242 [02:30<00:00, 68.25it/s]


[10/11] Formatting dataset...
Rows before dropna: 10255
Rows after dropna: 10124
[11/11] Extracting BERT Embeddings...


Extracting BERT Embeddings: 100%|██████████| 10124/10124 [06:23<00:00, 26.39it/s]


[DONE] Saving final unified dataset...
Removed temporary checkpoint file.
PIPELINE COMPLETE!
Final DataFrame Shape: (10124, 791)
Total Samples: 10124
Dataset successfully saved to: /home/basel/Work/python/ai_text_detection/data/processed/processed_misspelled_5.csv
